<div class="panel-academico portada-academica"><div class="etiqueta">Grupo 06 / Analítica educativa</div><h1>Cada estudiante,<br>una oportunidad</h1><p style="font-size:19px">Estimación de la nota final en educación secundaria</p><p>Un recorrido por los datos, la comparación de modelos y la simulación de perfiles académicos.</p><p style="margin-top:22px;font-size:12px;letter-spacing:1px">16 ETAPAS &nbsp; / &nbsp; 2 ESCENARIOS &nbsp; / &nbsp; ESCALA DE 0 A 20</p></div>

**Para comenzar:** selecciona el kernel de Python y utiliza **Run > Run All Cells**. Espera a que termine el entrenamiento para utilizar el simulador.

[Conjunto de datos](#etapa-3) &nbsp; / &nbsp; [Exploración](#etapa-6) &nbsp; / &nbsp; [Modelos](#etapa-8) &nbsp; / &nbsp; [Evaluación](#etapa-13) &nbsp; / &nbsp; [Simulador](#simulador)

Las columnas se traducen al cargarse. Los archivos originales de datos se conservan intactos.


##  Configuración del Entorno

In [ ]:
import sys
from pathlib import Path

ubicacion = Path().resolve()
RAIZ = next((carpeta for carpeta in [ubicacion, *ubicacion.parents, ubicacion / 'Trabajo_II_Unidad'] if (carpeta / 'src' / 'config.py').exists()), None)
if RAIZ is None:
    raise FileNotFoundError('Abre la carpeta que contiene src, notebooks y requisitos.txt en VS Code.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 130, 'font.family': 'DejaVu Sans'})
import seaborn as sns
sns.set_theme(style='whitegrid', palette='deep')

print(' Entorno configurado correctamente.')
print(f'   Raíz del proyecto: {RAIZ}')

from IPython.display import display
from src.interfaz import configurar_estilo
configurar_estilo()


---
<a id="etapa-1"></a>
<div class="seccion-cuaderno"><span>ETAPA 1 / 16</span><h2>Planteamiento del Problema</h2></div>

### 1.1 Contexto Institucional

En los sistemas educativos, la detección oportuna de estudiantes en riesgo de reprobación permite
diseñar intervenciones pedagógicas efectivas. El presente proyecto utiliza datos de estudiantes
de asignatura de **Lengua Portuguesa** en dos escuelas secundarias de Portugal para construir un
sistema de predicción de la **calificación final (`nota_final`)**.

### 1.2 Definición Formal del Problema

Se desea aprender una función de regresión:

$$\hat{nota_final} = f(predictores)$$

donde $predictores$ es el conjunto de variables socioeconómicas, de historial académico y de comportamiento,
y $\hat{nota_final} \in [0, 20]$ es la calificación final estimada.

### 1.3 Dos Escenarios de Intervención

| Escenario | Momento de Uso | Variables Disponibles | Utilidad |
|---|---|---|---|
| **Temprano** | Inicio del periodo escolar | Sin nota_primer_periodo ni nota_segundo_periodo | Alerta temprana para prevención |
| **Tardío** | Después del 2° periodo | Incluye nota_primer_periodo y nota_segundo_periodo | Maximiza precisión antes del cierre |

### 1.4 Justificación

- **Subestimar** la nota de un alumno en riesgo puede retrasar una intervención necesaria.
- **Sobreestimar** puede derivar en recursos desperdiciados en alumnos que no los necesitan.
- Separar los dos escenarios evita la confusión entre alerta temprana y predicción tardía.


---
<a id="etapa-2"></a>
<div class="seccion-cuaderno"><span>ETAPA 2 / 16</span><h2>Objetivos y Criterios de Éxito</h2></div>

### 2.1 Objetivo General

Construir un modelo de regresión supervisada que estime la calificación final `nota_final` con un
**Error Absoluto Medio (MAE) inferior a 1.5 puntos** sobre el conjunto reservado de prueba.

### 2.2 Objetivos Específicos

| N° | Objetivo |
|---|---|
| 1 | Análisis exploratorio completo y Tablas T1 y T2. |
| 2 | Construir flujos sin fuga de información reproducibles. |
| 3 | Comparar 4 algoritmos con CV de 5 pliegues. |
| 4 | Optimizar hiperparámetros de KNN y SVR con `GridSearchCV`. |
| 5 | Evaluar el modelo ganador **una sola vez** sobre el test reservado. |

### 2.3 Criterios de Éxito Preestablecidos

| Métrica | Umbral | Interpretación |
|---|---|---|
| **MAE (Test)** | < 1.5 pts | Margen de error tolerable en escala 0–20 |
| **R² (Test)** | > 0.70 | El modelo explica al menos 70% de la varianza |
| **Mejora vs. Baseline** | > 30% | Reducción de error frente a DummyRegressor |
| **CV estable** | MAE\_σ < 0.30 | Modelo sin sobreajuste severo entre pliegues |


---
<a id="etapa-3"></a>
<div class="seccion-cuaderno"><span>ETAPA 3 / 16</span><h2>Dataset y Diccionario de Datos (Tabla T1)</h2></div>

In [ ]:
from src.data_loader import cargar_datos, calcular_sha256
from src.eda import generar_tabla_t1_diccionario, generar_tabla_t2_calidad
from src.config import RUTA_DATOS_ORIGINALES, RUTA_ZIP_ORIGINAL, METADATOS_DATOS

datos = cargar_datos(RUTA_DATOS_ORIGINALES)

print('=' * 60)
print(' METADATOS DEL DATASET')
print('=' * 60)
for k, v in METADATOS_DATOS.items():
    print(f'  {k:25s}: {v}')

print(f'\n{"=" * 60}')
print(' CONTROL DE INTEGRIDAD SHA-256')
print(f'{"=" * 60}')
if RUTA_ZIP_ORIGINAL.exists():
    print(f'  ZIP : {calcular_sha256(RUTA_ZIP_ORIGINAL)}')
print(f'  CSV : {calcular_sha256(RUTA_DATOS_ORIGINALES)}')

print(f'\n  Dimensiones: {datos.shape[0]} registros x {datos.shape[1]} columnas')

from src.interfaz import mostrar_resumen
mostrar_resumen(datos)


In [ ]:
t1 = generar_tabla_t1_diccionario(datos)
print(' TABLA T1 - Diccionario de Variables')
display(t1[['Variable', 'Definición', 'Tipo de Dato', 'Valores / Rango', 'Rol',
           'Disp. Temprana', 'Disp. Tardía']].style
        .set_caption('Tabla T1: Diccionario Técnico de Variables')
        .map(lambda v: 'background-color: #fff3cd' if 'Objetivo' in str(v) else
                       ('background-color: #d4edda' if 'Predictor' in str(v) else ''),
             subset=['Rol']))


---
<a id="etapa-4"></a>
<div class="seccion-cuaderno"><span>ETAPA 4 / 16</span><h2>Separación predictores/objetivo y Definición de Escenarios</h2></div>

In [ ]:
from src.data_loader import construir_escenarios, clasificar_columnas

escenarios = construir_escenarios(datos)
predictores_temprano, objetivo = escenarios['temprano']
predictores_tardio, _ = escenarios['tardio']
cols_num_temp, cols_cat_temp = clasificar_columnas(predictores_temprano)
cols_num_tard, cols_cat_tard = clasificar_columnas(predictores_tardio)

resumen = pd.DataFrame([
    {'Escenario': ' Temprano', 'Vars Excluidas': 'nota_primer_periodo, nota_segundo_periodo y nota_final',
     'Predictores': predictores_temprano.shape[1], 'Num': len(cols_num_temp),
     'Cat': len(cols_cat_temp), 'Riesgo Fuga': ' Nulo'},
    {'Escenario': ' Tardío', 'Vars Excluidas': 'nota_final',
     'Predictores': predictores_tardio.shape[1], 'Num': len(cols_num_tard),
     'Cat': len(cols_cat_tard), 'Riesgo Fuga': ' Nulo (nota_final excluido)'},
])
display(resumen.set_index('Escenario').style.set_caption('Definición de Escenarios Temprano vs Tardío'))
print(f'\n Variable Objetivo: nota_final | Rango: {datos["nota_final"].min()}-{datos["nota_final"].max()} | '
      f'Media: {datos["nota_final"].mean():.2f} | Mediana: {datos["nota_final"].median():.1f}')


---
<a id="etapa-5"></a>
<div class="seccion-cuaderno"><span>ETAPA 5 / 16</span><h2>Análisis Exploratorio de Datos (EDA) y Tabla T2</h2></div>

In [ ]:
t2 = generar_tabla_t2_calidad(datos)
print(' TABLA T2 - Diagnóstico de Calidad')
display(t2[['Variable','Tipo','Nulos','% Nulos','Valores Únicos','Mínimo','Máximo','Media',
           'Anomalías Observadas']].style
        .set_caption('Tabla T2: Diagnóstico de Calidad del Dataset')
        .highlight_max(subset=['Nulos'], color='#f8d7da'))

print(f'\n  Nulos totales : {datos.isna().sum().sum()}')
print(f'  Duplicados    : {datos.duplicated().sum()}')
print(f'  Calidad       :  Excelente (sin nulos ni duplicados)')
print('  ℹ  El flujo conserva SimpleImputer para robustez en datos futuros.')


In [ ]:
from src.eda import generar_graficos_eda
figuras = generar_graficos_eda(datos, guardar=True)
for nombre, fig in figuras.items():
    display(fig)
plt.close('all')


###  Hallazgos Exploratorios Clave

| N° | Hallazgo | Implicación |
|---|---|---|
| 1 | **nota_segundo_periodo** tiene la correlación más fuerte con nota_final (r ≈ 0.92) | Ventaja predictiva del escenario tardío |
| 2 | **nota_primer_periodo** también fuertemente correlacionada (r ≈ 0.80) | Ambas notas parciales son predictores críticos |
| 3 | Mayor historial de reprobaciones (`asignaturas_reprobadas`) → nota_final más bajas | Variable relevante en ambos escenarios |
| 4 | Ausencias tienen correlación negativa débil (r ≈ -0.09) | Señal real pero baja |
| 5 | nota_final=0 en algunos estudiantes | Posible abandono; datos válidos que se conservan |


---
<a id="etapa-6"></a>
<div class="seccion-cuaderno"><span>ETAPA 6 / 16</span><h2>Partición Entrenamiento / Prueba</h2></div>

In [ ]:
from src.data_loader import realizar_particion_datos, obtener_particiones_escenario

indices_entrenamiento, indices_prueba = realizar_particion_datos(datos, guardar_indices=True)
predictores_entrenamiento_tardio, predictores_prueba_tardio, objetivo_entrenamiento, objetivo_prueba = obtener_particiones_escenario(
    datos, 'tardio', indices_entrenamiento, indices_prueba)
predictores_entrenamiento_temprano, predictores_prueba_temprano, _, _ = obtener_particiones_escenario(
    datos, 'temprano', indices_entrenamiento, indices_prueba)

assert set(indices_entrenamiento).isdisjoint(set(indices_prueba)), 'FUGA DETECTADA!'
assert len(set(indices_entrenamiento) | set(indices_prueba)) == len(datos)

tabla = pd.DataFrame([
    {'Partición': ' Entrenamiento', 'N': len(indices_entrenamiento),
     '%': f'{len(indices_entrenamiento)/len(datos)*100:.1f}%', 'Uso': 'Ajuste + CV + GridSearch'},
    {'Partición': ' Prueba (Test)', 'N': len(indices_prueba),
     '%': f'{len(indices_prueba)/len(datos)*100:.1f}%', 'Uso': 'Evaluación Única del ganador (Etapa 13)'},
])
display(tabla.set_index('Partición').style.set_caption('Partición 80/20 con random_state=42'))
print('   Integridad verificada: train ∩ test = ∅')
print('   El test NO se consultará hasta la Etapa 13.')


---
<a id="etapa-7"></a>
<div class="seccion-cuaderno"><span>ETAPA 7 / 16</span><h2>Preprocesamiento Sin Fuga de Información (Tabla T4)</h2></div>

In [ ]:
from src.preprocessing import (crear_preprocesador, generar_tabla_t4_preprocesamiento,
                                verificar_transformacion_datos_no_vistos)

t4 = generar_tabla_t4_preprocesamiento(cols_num_tard, cols_cat_tard, 'tardio')
print('  ESTRATEGIA DE PREPROCESAMIENTO\n')
for _, fila in t4.iterrows():
    print(f'   {fila["Grupo"]} ({fila["Cantidad"]} columnas)')
    print(f'     Imputación : {fila["Imputación"]}')
    print(f'     Escalado   : {fila["Escalado"]}')
    print(f'     Codif.     : {fila["Codificación"][:60]}...')
    print()

prep = crear_preprocesador(cols_num_tard, cols_cat_tard)
ok = verificar_transformacion_datos_no_vistos(prep, predictores_entrenamiento_tardio)
print(f'   Robustez ante datos no vistos verificada: {ok}')
print(f'   fit() solo en fold de entrenamiento dentro del Pipeline (anti-fuga)')


---
<a id="etapa-8"></a>
<div class="seccion-cuaderno"><span>ETAPA 8 / 16</span><h2>Fundamentos y Selección de Modelos Candidatos</h2></div>

In [ ]:
from src.models import generar_tabla_fundamentos_modelos, ensamblar_pipelines

t_fund = generar_tabla_fundamentos_modelos()
print(' FUNDAMENTOS TEÓRICOS DE LOS ALGORITMOS')
display(t_fund[['Algoritmo','Mecanismo Matemático','Sensibilidad al Escalado',
               'Fortalezas','Limitaciones','Hiperparámetros Clave']].style
        .set_caption('Justificación técnica de los 4 algoritmos seleccionados')
        .set_properties(**{'white-space':'pre-wrap','font-size':'11px'}))

flujos_tardio = ensamblar_pipelines(cols_num_tard, cols_cat_tard)
flujos_temprano = ensamblar_pipelines(cols_num_temp, cols_cat_temp)
print('\n PIPELINES ENSAMBLADOS:')
for nombre, flujo in flujos_tardio.items():
    print(f'  • {nombre:35s} → {type(flujo.named_steps["modelo"]).__name__}')


---
<a id="etapa-9"></a>
<div class="seccion-cuaderno"><span>ETAPA 9 / 16</span><h2>Entrenamiento Reproducible de los Pipelines</h2></div>

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
print(' CONFIGURACIÓN DE ENTRENAMIENTO REPRODUCIBLE')
print('=' * 55)
print(f'  Algoritmo de partición : KFold')
print(f'  N° de pliegues (folds) : 5')
print(f'  shuffle                : True')
print(f'  random_state           : 42')
print(f'\n  Métricas de evaluación interna:')
print(f'  • MAE  = (1/n) * sum(|y_i - y_hat_i|)')
print(f'  • RMSE = sqrt[(1/n) * sum((y_i - y_hat_i)^2)]')
print(f'  • R²   = 1 - SS_res / SS_tot')
print(f'\n  predictores_entrenamiento Tardío : {predictores_entrenamiento_tardio.shape}')
print(f'  predictores_entrenamiento Temprano: {predictores_entrenamiento_temprano.shape}')


---
<a id="etapa-10"></a>
<div class="seccion-cuaderno"><span>ETAPA 10 / 16</span><h2>Validación Cruzada de 5 Pliegues (Tabla T5)</h2></div>

In [ ]:
from src.evaluation import evaluar_validacion_cruzada

print(' Ejecutando CV 5-Fold - Escenario Tardío...')
t5_tard, res_tard = evaluar_validacion_cruzada(
    flujos_tardio, predictores_entrenamiento_tardio, objetivo_entrenamiento, nombre_escenario='tardio')

cols_show = ['Modelo','Fold 1 (MAE)','Fold 2 (MAE)','Fold 3 (MAE)',
             'Fold 4 (MAE)','Fold 5 (MAE)','MAE Medio','MAE Desv.','RMSE Medio','R2 Medio','Ranking MAE']
print('\n TABLA T5a - CV Escenario Tardío')
display(t5_tard[cols_show].style
        .set_caption('Tabla T5a: 5-Fold CV - Escenario Tardío')
        .background_gradient(subset=['MAE Medio'], cmap='RdYlGn_r')
        .background_gradient(subset=['R2 Medio'], cmap='RdYlGn'))


In [ ]:
print(' Ejecutando CV 5-Fold - Escenario Temprano...')
t5_temp, res_temp = evaluar_validacion_cruzada(
    flujos_temprano, predictores_entrenamiento_temprano, objetivo_entrenamiento, nombre_escenario='temprano')

print('\n TABLA T5b - CV Escenario Temprano')
display(t5_temp[cols_show].style
        .set_caption('Tabla T5b: 5-Fold CV - Escenario Temprano')
        .background_gradient(subset=['MAE Medio'], cmap='RdYlGn_r')
        .background_gradient(subset=['R2 Medio'], cmap='RdYlGn'))


In [ ]:
fig_cv, ejes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (t5, titulo) in zip(ejes, [
        (t5_tard, 'Escenario Tardío (con nota_primer_periodo y nota_segundo_periodo)'),
        (t5_temp, 'Escenario Temprano (sin nota_primer_periodo ni nota_segundo_periodo)')]):
    colores = ['#2ecc71' if i==0 else '#3498db' if i==1 else '#e67e22' if i==2 else '#e74c3c'
               for i in range(len(t5))]
    bars = ax.barh(t5['Modelo'], t5['MAE Medio'], xerr=t5['MAE Desv.'],
                   color=colores, edgecolor='black', capsize=4)
    for bar, (_, row) in zip(bars, t5.iterrows()):
        ax.text(bar.get_width()+0.02, bar.get_y()+bar.get_height()/2,
                f'MAE={row["MAE Medio"]:.3f}', va='center', fontsize=9, fontweight='bold')
    ax.set_title(f'MAE por Modelo - {titulo}', fontsize=11, weight='bold')
    ax.set_xlabel('MAE (puntos en escala 0-20)')
    ax.set_xlim(0, max(t5['MAE Medio'])*1.4)
plt.suptitle('Figura 9: Comparativa de MAE en Validación Cruzada 5-Fold', fontsize=13, weight='bold')
plt.tight_layout()
plt.savefig(str(RAIZ/'reports'/'figures'/'09_comparativa_cv.png'), dpi=200, bbox_inches='tight')
plt.show()


###  Análisis de Resultados de CV

**Escenario Tardío:**
- `LinearRegression` lidera con MAE = 0.863 y R² = 0.837, gracias a la fuerte linealidad nota_primer_periodo/nota_segundo_periodo → nota_final.
- `SVR` muestra alta estabilidad entre pliegues (σ = 0.077).
- `DummyRegressor` confirma el baseline sin aprendizaje (MAE ≈ 2.421).

**Escenario Temprano:**
- Sin nota_primer_periodo y nota_segundo_periodo, el problema es significativamente más difícil (todos los MAE aumentan ~2×).
- R² ≈ 0.30: las variables socioeconómicas explican solo el 30% de la varianza.


---
<a id="etapa-11"></a>
<div class="seccion-cuaderno"><span>ETAPA 11 / 16</span><h2>Ajuste de Hiperparámetros con GridSearchCV (Tabla T6)</h2></div>

In [ ]:
from src.evaluation import optimizar_hiperparametros
from src.config import REJILLA_KNN, REJILLA_SVR

print(' Mallas de búsqueda definidas:')
print('\n  KNN:')
for param, vals in REJILLA_KNN.items():
    print(f'    {param}: {vals}')
print('\n  SVR:')
for param, vals in REJILLA_SVR.items():
    print(f'    {param}: {vals}')
total_knn = 1
for v in REJILLA_KNN.values(): total_knn *= len(v)
total_svr = 1
for v in REJILLA_SVR.values(): total_svr *= len(v)
print(f'\n  Configuraciones KNN: {total_knn} x 5 folds = {total_knn*5} ajustes')
print(f'  Configuraciones SVR: {total_svr} x 5 folds = {total_svr*5} ajustes')
print('\n Ejecutando GridSearchCV (solo en predictores_entrenamiento)...')


In [ ]:
t6, busquedas_rejilla = optimizar_hiperparametros(
    flujos_tardio, predictores_entrenamiento_tardio, objetivo_entrenamiento, nombre_escenario='tardio')

print('\n TABLA T6 - Resultados de Optimización de Hiperparámetros')
display(t6.style
        .set_caption('Tabla T6: Optimización con GridSearchCV - Escenario Tardío')
        .background_gradient(subset=['Mejora (%)'], cmap='Greens'))

print('\n Mejores Hiperparámetros Encontrados:')
for nombre, gs in busquedas_rejilla.items():
    print(f'\n  {nombre}:')
    for param, valor in gs.best_params_.items():
        print(f'    {param}: {valor}')
    print(f'    MAE CV optimizado: {-gs.best_score_:.3f}')


---
<a id="etapa-12"></a>
<div class="seccion-cuaderno"><span>ETAPA 12 / 16</span><h2>Selección Formal del Modelo Ganador (Tabla T7)</h2></div>

>  **Principio Fundamental:** La selección se realiza **exclusivamente con resultados de CV**.
> El conjunto de prueba permanece sellado hasta la Etapa 13.


In [ ]:
from src.evaluation import generar_tabla_t7_decision

mae_lr = t5_tard.loc[t5_tard['Modelo']=='regresion_lineal','MAE Medio'].values[0]
mae_svr_opt = -busquedas_rejilla['regresion_vectores_soporte'].best_score_
mae_knn_opt = -busquedas_rejilla['vecinos_cercanos'].best_score_

if mae_svr_opt <= mae_lr:
    ganador_nombre = 'regresion_vectores_soporte'
    flujo_ganador = busquedas_rejilla['regresion_vectores_soporte'].best_estimator_
    mae_ganador = mae_svr_opt
else:
    ganador_nombre = 'regresion_lineal'
    flujo_ganador = flujos_tardio['regresion_lineal']
    mae_ganador = mae_lr

candidatos = [
    {'Candidato':'LinearRegression', 'MAE CV':round(float(mae_lr),3),
     'Interpretabilidad':'Alta', 'Costo Inferencia':'O(p)',
     'Veredicto':' GANADOR' if ganador_nombre=='regresion_lineal' else 'Finalista'},
    {'Candidato':'SVR Optimizado', 'MAE CV':round(float(mae_svr_opt),3),
     'Interpretabilidad':'Baja (kernel)', 'Costo Inferencia':'O(n_sv*p)',
     'Veredicto':' GANADOR' if ganador_nombre=='regresion_vectores_soporte' else 'Finalista'},
    {'Candidato':'KNN Optimizado', 'MAE CV':round(float(mae_knn_opt),3),
     'Interpretabilidad':'Media', 'Costo Inferencia':'O(n*p)',
     'Veredicto':'Descartado'},
    {'Candidato':'DummyRegressor', 'MAE CV':round(float(
         t5_tard.loc[t5_tard['Modelo']=='referencia_media','MAE Medio'].values[0]),3),
     'Interpretabilidad':'N/A', 'Costo Inferencia':'O(1)',
     'Veredicto':'Referencia Mínima'},
]
t7 = generar_tabla_t7_decision(candidatos, nombre_escenario='tardio')
print(' TABLA T7 - Decisión Formal del Modelo Ganador')
display(t7.style.set_caption('Tabla T7: Selección del Ganador - SIN tocar Test')
        .apply(lambda col: ['background-color: #d4edda; font-weight: bold'
                            if ' GANADOR' in str(v) else '' for v in col], axis=0))
print(f'\n MODELO SELECCIONADO: {ganador_nombre.upper()}')
print(f'   MAE en CV: {mae_ganador:.3f} puntos')
print(f'    Test reservado — NO consultado hasta la Etapa 13.')


---
<a id="etapa-13"></a>
<div class="seccion-cuaderno"><span>ETAPA 13 / 16</span><h2>Evaluación Final en el Conjunto de Prueba Reservado (Tabla T8)</h2></div>

>  **Única evaluación sobre el 20% reservado. Esta operación no puede repetirse con ajuste posterior.**


In [ ]:
from src.evaluation import evaluar_test_final

print(' ABRIENDO CONJUNTO DE PRUEBA (TEST SET) — Operación única')
print(f'   N de estudiantes en test: {len(objetivo_prueba)}')

t8, datos_10, figs_eval = evaluar_test_final(
    flujo_ganador=flujo_ganador,
    flujo_referencia=flujos_tardio['referencia_media'],
    predictores_entrenamiento=predictores_entrenamiento_tardio, objetivo_entrenamiento=objetivo_entrenamiento,
    predictores_prueba=predictores_prueba_tardio, objetivo_prueba=objetivo_prueba,
    nombre_modelo=ganador_nombre,
    nombre_escenario='tardio',
)
print('\n TABLA T8 - Evaluación en Conjunto de Prueba')
display(t8.style
        .set_caption('Tabla T8: Resultados Finales en Test (N=130 estudiantes)')
        .background_gradient(subset=['MAE Test'], cmap='RdYlGn_r')
        .background_gradient(subset=['R2 Test'], cmap='RdYlGn'))


In [ ]:
flujo_ganador.fit(predictores_entrenamiento_tardio, objetivo_entrenamiento)
predicciones_prueba = flujo_ganador.predict(predictores_prueba_tardio)

print('\n TABLA T8b - 10 Ejemplos Real vs Predicho')
display(datos_10.style
        .set_caption('Comparación Real vs Predicho en los primeros 10 estudiantes de prueba')
        .background_gradient(subset=['Error_Absoluto'], cmap='Oranges'))


In [ ]:
for nombre_fig, fig in figs_eval.items():
    display(fig)
plt.close('all')


###  Interpretación de los Resultados en Test

| Métrica | Valor Obtenido | Umbral | Superado |
|---|---|---|---|
| **MAE** | ~0.70 pts | < 1.5 pts |  Sí |
| **RMSE** | ~1.15 pts | < 2.0 pts |  Sí |
| **R²** | ~0.86 | > 0.70 |  Sí |
| **Mejora vs. Baseline** | ~70% | > 30% |  Sí |

- El modelo se equivoca en promedio en **menos de 1 punto** sobre una escala de 0 a 20.
- Explica el **86% de la varianza** en datos no vistos.
- Los residuales son aproximadamente simétricos alrededor de cero: **sin sesgo sistemático**.


---
<a id="etapa-14"></a>
<div class="seccion-cuaderno"><span>ETAPA 14 / 16</span><h2>Análisis Crítico, Riesgos y Limitaciones</h2></div>

In [ ]:
residuales = objetivo_prueba.values - predicciones_prueba
sobreest = residuales[residuales < 0]
subeest  = residuales[residuales > 0]

print('  ANÁLISIS DEL COSTO ASIMÉTRICO DEL ERROR')
print('=' * 55)
print(f'\n  Sobreestimaciones (riesgo no-intervención):')
print(f'    N={len(sobreest)} | Error medio={abs(sobreest.mean()):.2f} pts | Máx={abs(sobreest.min()):.2f} pts')
print(f'      Alumno en riesgo NO recibe apoyo a tiempo.')
print(f'\n  Subestimaciones (riesgo de sobreintervención):')
print(f'    N={len(subeest)} | Error medio={subeest.mean():.2f} pts | Máx={subeest.max():.2f} pts')
print(f'    ℹ  Recursos aplicados a alumnos que no los necesitan.')
print(f'\n  Sesgo general: {residuales.mean():+.4f} pts (0=sin sesgo ideal)')


In [ ]:
fig_riesgo, ax = plt.subplots(figsize=(9, 5))
sns.histplot(residuales, bins=18, kde=True, color='#3498db', edgecolor='k', ax=ax)
ax.axvline(0, color='black', linewidth=2, label='Error Cero (ideal)')
ax.axvline(residuales.mean(), color='red', linewidth=1.5, linestyle='--',
           label=f'Sesgo Observado ({residuales.mean():+.3f} pts)')
ax.set_title('Figura 10: Análisis del Riesgo Asimétrico - Distribución de Residuales',
             fontsize=12, weight='bold')
ax.set_xlabel('Residual = G3_Real - G3_Predicho (puntos)')
ax.set_ylabel('Frecuencia')
ax.legend()
plt.tight_layout()
plt.savefig(str(RAIZ/'reports'/'figures'/'10_riesgo_asimetrico.png'), dpi=200, bbox_inches='tight')
plt.show()


###  Tabla de Riesgos y Limitaciones

| Riesgo / Limitación | Severidad | Medida Adoptada |
|---|---|---|
| Dependencia de nota_primer_periodo y nota_segundo_periodo | Alta | Complementar con escenario temprano |
| Sobreestimación de nota | Alta | Ajustar umbral de alerta institucional |
| Generalización a otras asignaturas | Media | Reentrenar con datos de Matemáticas u otras |
| Extremos de distribución (nota_final=0 o nota_final=20) | Media | Monitorear error en colas de la distribución |
| Sesgos socioeconómicos | Media | Monitorear métricas por grupos demográficos |
| Deriva del dataset | Baja/Media | Recalibrar anualmente con datos actuales |


---
<a id="etapa-15"></a>
<div class="seccion-cuaderno"><span>ETAPA 15 / 16</span><h2>Demostración de Nuevas Predicciones (Tabla T9)</h2></div>

In [ ]:
from src.predict import cargar_pipeline_entrenado, generar_cinco_casos_nuevos, realizar_predicciones_nuevas

print(' PREDICCIONES SOBRE 5 NUEVOS CASOS HIPÓTETICOS')
print('  El flujo NO se reentrena. Inferencia directa.')

modelo = cargar_pipeline_entrenado()
t9 = realizar_predicciones_nuevas(modelo, escenario='tardio')
display(t9.style
        .set_caption('Tabla T9: 5 Perfiles Hipotéticos de Estudiantes')
        .apply(lambda col: [
            'background-color: #d4edda; color: #155724; font-weight: bold' if 'Aprobado' in str(v)
            else ('background-color: #f8d7da; color: #721c24; font-weight: bold' if 'Riesgo' in str(v)
            else '') for v in col], axis=0, subset=['Estado Proyectado']))


In [ ]:
datos_casos = generar_cinco_casos_nuevos(escenario='tardio')
predicciones = modelo.predict(datos_casos.drop(columns=['Caso','Perfil_Cualitativo']))

fig_casos, ax = plt.subplots(figsize=(10, 5))
colores = ['#27ae60' if p >= 10 else '#e74c3c' for p in predicciones]
bars = ax.bar(datos_casos['Caso'].str.replace('Caso ','C'), predicciones,
              color=colores, edgecolor='black')
ax.axhline(10, color='orange', linestyle='--', linewidth=2, label='Umbral aprobación (10 pts)')
ax.axhline(datos['nota_final'].mean(), color='blue', linestyle=':', linewidth=1.5,
           label=f'Media histórica (nota_final={datos["nota_final"].mean():.1f})')
for bar, pred in zip(bars, predicciones):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'{pred:.1f}', ha='center', fontweight='bold', fontsize=10)
ax.set_ylim(0, 22)
ax.set_title('Figura 11: Calificaciones Estimadas para los 5 Nuevos Perfiles',
             fontsize=12, weight='bold')
ax.set_ylabel('nota_final Estimado (escala 0-20)')
ax.legend()
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(str(RAIZ/'reports'/'figures'/'11_nuevas_predicciones.png'), dpi=200, bbox_inches='tight')
plt.show()


---
<a id="etapa-16"></a>
<div class="seccion-cuaderno"><span>ETAPA 16 / 16</span><h2>Conclusiones, Mejoras y Recomendación de Despliegue</h2></div>

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.svm import SVR
from src.preprocessing import crear_preprocesador
from src.config import REJILLA_SVR

# Evaluar mejor SVR en escenario temprano
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)
pipe_svr_temp = Pipeline([
    ('preprocesamiento', crear_preprocesador(cols_num_temp, cols_cat_temp)),
    ('modelo', SVR()),
])
grid_temp = GridSearchCV(pipe_svr_temp, REJILLA_SVR, cv=kf5,
                         scoring='neg_mean_absolute_error', n_jobs=-1)
grid_temp.fit(predictores_entrenamiento_temprano, objetivo_entrenamiento)
predicciones_temp = grid_temp.best_estimator_.predict(predictores_prueba_temprano)

mae_tard = mean_absolute_error(objetivo_prueba, predicciones_prueba)
mae_temp = mean_absolute_error(objetivo_prueba, predicciones_temp)
mae_dum  = mean_absolute_error(objetivo_prueba, [objetivo_entrenamiento.mean()]*len(objetivo_prueba))

tabla_final = pd.DataFrame([
    {'Escenario':' Temprano', 'Modelo':'SVR Opt.',
     'MAE Test':round(mae_temp,3), 'RMSE Test':round(float(mean_squared_error(objetivo_prueba,predicciones_temp)**0.5),3),
     'R² Test':round(r2_score(objetivo_prueba,predicciones_temp),3),
     'Mejora vs Baseline':f'{((mae_dum-mae_temp)/mae_dum*100):.1f}%',
     'Uso Recomendado':'Intervención preventiva inicio de año'},
    {'Escenario':' Tardío', 'Modelo':'SVR Opt.',
     'MAE Test':round(mae_tard,3), 'RMSE Test':round(float(mean_squared_error(objetivo_prueba,predicciones_prueba)**0.5),3),
     'R² Test':round(r2_score(objetivo_prueba,predicciones_prueba),3),
     'Mejora vs Baseline':f'{((mae_dum-mae_tard)/mae_dum*100):.1f}%',
     'Uso Recomendado':'Pronóstico de cierre antes del acta'},
])
print(' TABLA RESUMEN FINAL - Ambos Escenarios en Test (N=130)')
display(tabla_final.set_index('Escenario').style
        .set_caption('Comparativa Final Temprano vs Tardío')
        .background_gradient(subset=['MAE Test'], cmap='RdYlGn_r')
        .background_gradient(subset=['R² Test'], cmap='RdYlGn'))


###  Conclusiones Fundamentadas

1. **La regresión supervisada es adecuada.** nota_final es continua y los modelos superaron con holgura el umbral de éxito (MAE < 1.5 pts en test).

2. **nota_primer_periodo y nota_segundo_periodo son predictores de primer orden.** La correlación de nota_segundo_periodo con nota_final es r ≈ 0.92; el escenario tardío tiene una ventaja predictiva sustancial (~0.70 vs ~1.80 pts de MAE).

3. **SVR con kernel lineal y C=0.1 fue el modelo ganador.** Supera a LinearRegression tras la optimización y logra MAE ≈ 0.70 pts con R² ≈ 0.86 en test.

4. **El aislamiento estricto del test garantiza estimaciones confiables.** Todos los parámetros se determinaron sobre el 80% de entrenamiento, sin optimismo artificial.

5. **El escenario temprano tiene alto valor institucional.** MAE de ~1.8 pts con datos desde la matrícula permite identificar alumnos en riesgo antes de cualquier evaluación.

---

###  Mejoras Técnicas Priorizadas

| Prioridad | Mejora | Impacto Esperado |
|---|---|---|
| **Alta** | Incorporar `RandomForest` y `GradientBoosting` como candidatos adicionales | Captura de interacciones no lineales; posible MAE < 1.3 en temprano |
| **Alta** | Aplicar `RandomizedSearchCV` con distribuciones continuas para SVR | Exploración más eficiente del espacio de hiperparámetros |
| **Media** | Analizar importancia de características (permutation importance) | Identificar las 5 variables más críticas por escenario |
| **Media** | Equilibrar la distribución de nota_final en los extremos (0-4 y 18-20) | Reducción del sesgo hacia la media histórica |
| **Baja** | Desplegar el flujo como API REST (FastAPI + joblib) | Integración directa con sistemas de gestión académica |

---

###  Recomendación de Despliegue

```python
import joblib
flujo = joblib.load('models/modelo_final_tardio_support_vector_regression.joblib')
nota_estimada = flujo.predict(predictores_nuevo_estudiante)  # Sin reentrenamiento
if nota_estimada[0] < 10:
    activar_protocolo_intervencion()
```

**Ciclo de mantenimiento recomendado:** Recalibrar al inicio de cada año escolar con datos actualizados.


In [ ]:
from src.config import DIRECTORIO_TABLAS, DIRECTORIO_FIGURAS, DIRECTORIO_MODELOS

print('=' * 70)
print(' NOTEBOOK DE 16 ETAPAS COMPLETADO EXITOSAMENTE')
print('=' * 70)
print('\n   Tablas generadas en reports/tables/:')
for t in sorted(DIRECTORIO_TABLAS.glob('*.csv')): print(f'     {t.name}')
print('\n   Figuras generadas en reports/figures/:')
for f in sorted(DIRECTORIO_FIGURAS.glob('*.png')): print(f'     {f.name}')
print('\n   Modelos serializados en models/:')
for m in sorted(DIRECTORIO_MODELOS.glob('*.joblib')): print(f'     {m.name}')


<a id="simulador"></a>
<div class="seccion-cuaderno"><span>EXPLORACIÓN INTERACTIVA</span><h2>Simulador de nota final</h2></div>
Completa las cuatro secciones del perfil y calcula la estimación. Ejecuta primero las etapas anteriores.

In [ ]:
from src.interfaz import construir_panel
campos_perfil, boton_estimar, salida_estimacion = construir_panel(
    flujo_ganador, datos.drop(columns=['nota_final']))
